# Temporal LCI Routing Comparison

This notebook compares two temporal routing choices for the foreground case-study inventories:

- **Foreground-only routing:** expand one level from the functional unit, then solve the remaining demand through the annual matrices.
- **Adaptive routing:** keep expanding branches whose static score potential is large enough to matter, up to a depth cap.

For each case, the notebook runs a static LCA, both temporal routing cases, and a simple annual/cumulative plot. The notebook intentionally favors readability over runtime optimization.

## Configuration

Edit this cell to choose cases, methods, routing parameters, and output locations.

In [ ]:
from __future__ import annotations

import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from datapackage import Package
from IPython.display import display
from openpyxl import load_workbook

from trails import Trails, get_lcia_method_names

HERE = Path.cwd()
PUBLICATION_DIR = HERE if (HERE / "LCIs").exists() else HERE / "dev" / "publication"
REPO_ROOT = PUBLICATION_DIR.parents[1]

DATAPACKAGE = PUBLICATION_DIR / "trails_remind_SSP2-PkBudg1000.zip"

LCI_DIR = PUBLICATION_DIR / "LCIs"
OUTPUT_DIR = PUBLICATION_DIR / "notebook_runs" / "temporal_lci_routing_comparison_clean"
RESULTS_CSV = OUTPUT_DIR / "routing_comparison_scores.csv"

CASE_KEYS = [
    "bev",
    "polyol",
    "marine",
    "daccs"
]
REFERENCE_YEAR = 2025
EI_VERSION = "3.12"

METHOD_PREFIX = "EF v3.1 - "

EXPECTED_INVENTORY_ACTIVITIES = {
    "lci-case-study-ccu_polyol_delayed_release.xlsx": {
        "polyol precursor production from captured CO2",
        "treatment of polyol, incineration",
    },
    "lci-case-study-daccs_storage_risk.xlsx": {
        "carbon dioxide, captured, with a solvent-based direct air capture system, 1MtCO2",
    },
    "lci-case-study-marine_fuel_switch.xlsx": {
        "marine freight service, temporal fuel transition",
    },
    "lci-pass_cars.xlsx": {
        "transport, passenger, car, diesel",
        "transport, passenger, car, battery electric",
        "heating, from wood logs stove",
        "heating, from heat pump",
    },
}

INVENTORY_PATHS = [
    LCI_DIR / filename for filename in sorted(EXPECTED_INVENTORY_ACTIVITIES)
]

DEMAND_AMOUNT_BY_CASE = {
    "bev": 150_000.0,
    "polyol": 50_000_000_000.0,
    "marine": 180_000_000_000.0,
    "daccs": 20_000_000_000.0,
}

FOREGROUND_DEPTH = 1
ADAPTIVE_RELATIVE_SCORE_CUTOFF = 1e-4
ADAPTIVE_MIN_DEPTH = 1
ADAPTIVE_MAX_DEPTH = None
ROUTING_MIN_AMOUNT = 0.0
SOLVER_MODE = "iterative"
FALLBACK_SOLVER_MODE = "direct"
SHOW_PROGRESS = False

INTERPOLATION_START_YEAR_OFFSET = -20
INTERPOLATION_END_YEAR_OFFSET = 20

PLOT_TOP_ACTIVITIES = 8
PLOT_YEAR_START = 1986
PLOT_YEAR_END = 2055

WIDTH = 640
HEIGHT = 504
PNG_SCALE = 3
WRITE_PNG = True
WRITE_HTML = False
STACKED_AREA_MODES = [False, True]

LEGEND_COLUMNS = 2
LEGEND_MAX_ROWS = 5
LEGEND_LABEL_MAX_CHARS = 50
X_TICK_START = 1990
X_TICK_YEARS = 10

ROUTING_CASE_LABELS = {
    "foreground": "Foreground only",
    "adaptive": "Adaptive routing",
}
ROUTING_CASE_COLORS = {
    "foreground": "#FFA500",
    "adaptive": "#FF0000",
    "static": "#dc2626",
}


## Small Helpers

These helpers are intentionally plain. They validate inputs, find imported activities by metadata, reduce score arrays to annual series, and make concise labels.

In [ ]:
@dataclass(frozen=True)
class ActivityDef:
    name: str
    reference_product: str
    location: str


@dataclass
class RoutingResult:
    case_key: str
    routing: str
    activity_index: int
    activity_label: str
    method: str
    static_score: float
    scores: Any
    annual: pd.Series
    cumulative: pd.Series
    routing_seconds: float
    lca_seconds: float
    solver_used: str
    graph_nodes: int
    graph_edges: int
    deepest_level: int


ACTIVITY_BY_CASE = {
    "bev": ActivityDef(
        "transport, passenger, car, battery electric",
        "transport, passenger, car",
        "RER",
    ),
    "polyol": ActivityDef(
        "polyol precursor production from captured CO2",
        "polyol precursor",
        "RER",
    ),
    "marine": ActivityDef(
        "marine freight service, temporal fuel transition",
        "transport service",
        "RER",
    ),
    "daccs": ActivityDef(
        "carbon dioxide, captured, with a solvent-based direct air capture system, 1MtCO2",
        "carbon dioxide, captured",
        "Europe",
    ),
}


def inventory_activity_names(path: Path) -> set[str]:
    workbook = load_workbook(path, data_only=True, read_only=True)
    names: set[str] = set()
    try:
        for worksheet in workbook.worksheets:
            for row in worksheet.iter_rows(values_only=True):
                values = [cell for cell in row if cell is not None]
                if len(values) >= 2 and str(values[0]).strip() == "Activity":
                    names.add(str(values[1]).strip())
    finally:
        workbook.close()
    return names


def validate_inventory_inputs() -> None:
    paths_by_name = {path.name: path for path in INVENTORY_PATHS}
    missing_files = sorted(set(EXPECTED_INVENTORY_ACTIVITIES) - set(paths_by_name))
    if missing_files:
        raise FileNotFoundError(
            "Missing publication inventory files:\n" + "\n".join(missing_files)
        )

    problems: list[str] = []

    for filename, expected_names in EXPECTED_INVENTORY_ACTIVITIES.items():
        actual_names = inventory_activity_names(paths_by_name[filename])
        missing_names = sorted(expected_names - actual_names)
        if missing_names:
            problems.append(
                f"{filename} is not the expected Figure 5 input; "
                f"missing activities: {missing_names}"
            )
    if problems:
        raise ValueError("\n".join(problems))


def validate_configuration() -> None:
    missing_paths = [path for path in [DATAPACKAGE, *INVENTORY_PATHS] if not path.exists()]
    if missing_paths:
        raise FileNotFoundError("Missing input files:\n" + "\n".join(map(str, missing_paths)))

    validate_inventory_inputs()

    unknown_cases = [key for key in CASE_KEYS if key not in ACTIVITY_BY_CASE]
    if unknown_cases:
        raise ValueError(f"Unknown CASE_KEYS: {unknown_cases}")


def get_available_methods(trails: Trails) -> list[str]:
    if hasattr(trails, "get_available_methods"):
        try:
            return list(trails.get_available_methods(ei_version=EI_VERSION))
        except TypeError:
            return list(trails.get_available_methods())
    if hasattr(trails, "list_lcia_methods"):
        return list(trails.list_lcia_methods(ei_version=EI_VERSION))
    return list(get_lcia_method_names(ei_version=EI_VERSION))


def get_ef_v31_methods(trails: Trails) -> list[str]:
    methods = [
        method
        for method in get_available_methods(trails)
        if str(method).startswith(METHOD_PREFIX)
    ]
    if not methods:
        raise ValueError(f"No available methods start with {METHOD_PREFIX!r}")
    return sorted(methods)

def clean_text(value: object) -> str:
    return "" if value is None else str(value).strip()


def matches_activity(meta: dict[str, Any], target: ActivityDef) -> bool:
    return (
        clean_text(meta.get("name")) == target.name
        and clean_text(meta.get("reference product")) == target.reference_product
        and clean_text(meta.get("location")) == target.location
    )


def find_activity_index(trails: Trails, target: ActivityDef) -> int:
    for metadata_by_index in trails.activity_indices.values():
        for index, meta in metadata_by_index.items():
            if matches_activity(meta, target):
                return int(index)
    raise ValueError(
        "Could not find activity: "
        f"{target.name} | {target.reference_product} | {target.location}"
    )


def activity_label(trails: Trails, activity_index: int) -> str:
    for metadata_by_index in trails.activity_indices.values():
        meta = metadata_by_index.get(int(activity_index))
        if meta:
            return " | ".join(
                part
                for part in [
                    clean_text(meta.get("name")),
                    clean_text(meta.get("reference product")),
                    clean_text(meta.get("location")),
                ]
                if part
            )
    return f"activity {activity_index}"


def graph_size(trails: Trails) -> tuple[int, int]:
    graph = getattr(trails, "graph", None)
    if graph is None:
        return 0, 0
    return len(graph.nodes), len(graph.edges)


def deepest_graph_level(trails: Trails) -> int:
    graph = getattr(trails, "graph", None)
    if graph is None:
        return 0
    depths = [int(data.get("depth", 0)) for _node, data in graph.nodes(data=True)]
    return max(depths, default=0)


def static_score_as_float(score: object) -> float:
    values = np.asarray(score, dtype=float).ravel()
    if values.size == 0:
        raise ValueError("Static score is empty")
    return float(values[0])


def dense_1d(data: Any) -> np.ndarray:
    raw = data.data if hasattr(data, "data") else data
    if hasattr(raw, "todense"):
        return np.asarray(raw.todense(), dtype=float).ravel()
    return np.asarray(data, dtype=float).ravel()


def select_method(scores: Any, method: str) -> Any:
    if "method" not in scores.dims:
        return scores
    methods = [str(value) for value in scores.coords["method"].values.tolist()]
    return scores.isel(method=methods.index(method), drop=True)


def reduce_scores(scores: Any, keep_dims: tuple[str, ...]) -> Any:
    kept = [dim for dim in keep_dims if dim in scores.dims]
    reduce_dims = [dim for dim in scores.dims if dim not in kept]
    reduced = scores.sum(dim=reduce_dims) if reduce_dims else scores
    return reduced.transpose(*kept) if tuple(reduced.dims) != tuple(kept) else reduced


def annual_total_series(scores: Any, method: str) -> pd.Series:
    by_year = reduce_scores(select_method(scores, method), ("year",))
    years = np.asarray(by_year.coords["year"].values, dtype=int)
    return pd.Series(dense_1d(by_year), index=years).sort_index()


def root_activity_frame(scores: Any, method: str) -> pd.DataFrame:
    by_root_year = reduce_scores(select_method(scores, method), ("root activity", "year"))
    root_ids = np.asarray(by_root_year.coords["root activity"].values, dtype=int)
    years = np.asarray(by_root_year.coords["year"].values, dtype=int)
    values = np.asarray(by_root_year.data.todense() if hasattr(by_root_year.data, "todense") else by_root_year.values, dtype=float)
    return pd.DataFrame(values, index=root_ids, columns=years)


def slug(value: str, max_length: int = 120) -> str:
    text = "".join(char.lower() if char.isalnum() else "_" for char in str(value))
    text = "_".join(part for part in text.split("_") if part)
    return (text or "item")[:max_length]

## Load Data

This loads the scenario datapackage, interpolates it to annual matrices, and imports only the four explicitly selected Figure 5 foreground inventory workbooks. Before running, the notebook checks that these files exist and contain the expected activity names. These checks do not verify exchange amounts or other inventory details, and do not guarantee that the inputs match those used for the manuscript figures.

In [ ]:
validate_configuration()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loading datapackage: {DATAPACKAGE}")
trails = Trails(
    Package(str(DATAPACKAGE)),
    interpolate_annual=True,
    cache_interpolation=True,
    interpolation_start_year_offset=INTERPOLATION_START_YEAR_OFFSET,
    interpolation_end_year_offset=INTERPOLATION_END_YEAR_OFFSET,
    ei_version=EI_VERSION,
)

EF_METHODS = get_ef_v31_methods(trails)
print(f"Selected {len(EF_METHODS)} methods starting with {METHOD_PREFIX!r}")

print("Importing foreground inventories:")
for path in INVENTORY_PATHS:
    print(f"  {path.name}")
trails.import_excel_inventory([str(path) for path in INVENTORY_PATHS])

activity_index_by_case = {
    key: find_activity_index(trails, ACTIVITY_BY_CASE[key])
    for key in CASE_KEYS
}
activity_index_by_case

## Run Static and Temporal LCA

Each case is run once with a fixed foreground depth and once with adaptive routing. Both temporal runs use root attribution so we can see which branches contribute over time.

In [ ]:
def static_scores_by_method(score: object, methods: list[str]) -> dict[str, float]:
    values = np.asarray(score, dtype=float).ravel()
    if values.size != len(methods):
        raise ValueError(
            f"Expected {len(methods)} static scores, got {values.size}."
        )
    return {method: float(value) for method, value in zip(methods, values)}


def run_temporal_routing_case(
    *,
    case_key: str,
    routing: str,
    activity_index: int,
    methods: list[str],
    amount: float,
    static_scores: dict[str, float],
) -> dict[str, RoutingResult]:
    if routing == "foreground_depth_1":
        routing_kwargs = dict(max_depth=FOREGROUND_DEPTH, adaptive_relative_score_cutoff=None)
    elif routing == "adaptive":
        routing_kwargs = dict(
            max_depth=ADAPTIVE_MAX_DEPTH,
            adaptive_relative_score_cutoff=ADAPTIVE_RELATIVE_SCORE_CUTOFF,
            adaptive_methods=methods,
            adaptive_ei_version=EI_VERSION,
            adaptive_min_depth=ADAPTIVE_MIN_DEPTH,
        )
    else:
        raise ValueError(f"Unknown routing case: {routing}")

    t0 = time.perf_counter()
    trails.temporal_routing(
        start_year=REFERENCE_YEAR,
        start_act_idx=activity_index,
        amount=amount,
        min_amount=ROUTING_MIN_AMOUNT,
        show_progress=SHOW_PROGRESS,
        attribute_to_roots=True,
        **routing_kwargs,
    )
    routing_seconds = time.perf_counter() - t0
    nodes, edges = graph_size(trails)
    deepest_level = deepest_graph_level(trails)

    def run_lca_with_solver(solver_mode: str) -> None:
        trails.lca(
            methods=methods,
            ei_version=EI_VERSION,
            show_progress=SHOW_PROGRESS,
            attribute_to_roots=True,
            compute_score=True,
            store_inventory=False,
            solver_mode=solver_mode,
        )

    solver_used = SOLVER_MODE
    t0 = time.perf_counter()
    try:
        run_lca_with_solver(SOLVER_MODE)
    except RuntimeError as error:
        can_retry_direct = (
            "GMRES failed to converge" in str(error)
            and str(FALLBACK_SOLVER_MODE).lower() == "direct"
            and str(SOLVER_MODE).lower() != "direct"
        )
        if not can_retry_direct:
            raise
        print(
            "      iterative solver did not converge; "
            "retrying this LCA with solver_mode='direct'"
        )
        solver_used = "direct"
        run_lca_with_solver("direct")
    lca_seconds = time.perf_counter() - t0

    scores = reduce_scores(trails.scores, ("method", "root activity", "year")).copy(deep=True)
    out: dict[str, RoutingResult] = {}
    for method in methods:
        annual = annual_total_series(scores, method)
        out[method] = RoutingResult(
            case_key=case_key,
            routing=routing,
            activity_index=activity_index,
            activity_label=activity_label(trails, activity_index),
            method=method,
            static_score=static_scores[method],
            scores=scores,
            annual=annual,
            cumulative=annual.cumsum(),
            routing_seconds=routing_seconds,
            lca_seconds=lca_seconds,
            solver_used=solver_used,
            graph_nodes=nodes,
            graph_edges=edges,
            deepest_level=deepest_level,
        )
    return out


results: dict[tuple[str, str, str], RoutingResult] = {}
rows: list[dict[str, Any]] = []

for case_key in CASE_KEYS:
    activity_index = activity_index_by_case[case_key]
    amount = DEMAND_AMOUNT_BY_CASE[case_key]
    label = activity_label(trails, activity_index)

    print(f"\n{case_key}: {label}")
    print(f"  amount={amount:g}")
    print(f"  methods={len(EF_METHODS)} EF v3.1 indicators")

    t0 = time.perf_counter()
    trails.static_lca(
        year=REFERENCE_YEAR,
        act_idx=activity_index,
        methods=EF_METHODS,
        amount=amount,
        ei_version=EI_VERSION,
    )
    static_scores = static_scores_by_method(trails.static_score, EF_METHODS)
    print(f"  static scores computed ({time.perf_counter() - t0:.1f}s)")

    for routing in ("foreground_depth_1", "adaptive"):
        routed_results = run_temporal_routing_case(
            case_key=case_key,
            routing=routing,
            activity_index=activity_index,
            methods=EF_METHODS,
            amount=amount,
            static_scores=static_scores,
        )
        for method, result in routed_results.items():
            results[(case_key, method, routing)] = result
        first_result = next(iter(routed_results.values()))
        print(
            f"  {routing}: depth={first_result.deepest_level}, "
            f"nodes={first_result.graph_nodes}, edges={first_result.graph_edges}, "
            f"solver={first_result.solver_used}, routing={first_result.routing_seconds:.1f}s, "
            f"lca={first_result.lca_seconds:.1f}s"
        )

    for method in EF_METHODS:
        foreground = results[(case_key, method, "foreground_depth_1")]
        adaptive = results[(case_key, method, "adaptive")]
        rows.append(
            {
                "case": case_key,
                "activity_index": activity_index,
                "activity": label,
                "amount": amount,
                "method": method,
                "static_score": static_scores[method],
                "temporal_cumulative_foreground_depth_1": foreground.cumulative.iloc[-1],
                "temporal_cumulative_adaptive": adaptive.cumulative.iloc[-1],
                "adaptive_minus_foreground": adaptive.cumulative.iloc[-1] - foreground.cumulative.iloc[-1],
                "adaptive_relative_to_foreground": (
                    np.nan
                    if foreground.cumulative.iloc[-1] == 0
                    else (adaptive.cumulative.iloc[-1] - foreground.cumulative.iloc[-1]) / foreground.cumulative.iloc[-1]
                ),
                "graph_nodes_foreground_depth_1": foreground.graph_nodes,
                "graph_nodes_adaptive": adaptive.graph_nodes,
                "graph_edges_foreground_depth_1": foreground.graph_edges,
                "graph_edges_adaptive": adaptive.graph_edges,
                "deepest_level_foreground_depth_1": foreground.deepest_level,
                "deepest_level_adaptive": adaptive.deepest_level,
                "solver_used_foreground_depth_1": foreground.solver_used,
                "solver_used_adaptive": adaptive.solver_used,
                "routing_seconds_foreground_depth_1": foreground.routing_seconds,
                "routing_seconds_adaptive": adaptive.routing_seconds,
                "lca_seconds_foreground_depth_1": foreground.lca_seconds,
                "lca_seconds_adaptive": adaptive.lca_seconds,
            }
        )

summary = pd.DataFrame(rows)
summary.to_csv(RESULTS_CSV, index=False)
display(summary)
print(f"Wrote {RESULTS_CSV}")

## Plot the Comparison

The annual areas come from the adaptive run because it has the richer root attribution. The cumulative lines compare foreground-only routing and adaptive routing directly.

In [ ]:
def top_root_contributions(result: RoutingResult) -> pd.DataFrame:
    frame = root_activity_frame(result.scores, result.method)
    totals = frame.abs().sum(axis=1).sort_values(ascending=False)
    top_roots = list(totals.head(PLOT_TOP_ACTIVITIES).index)
    plotted = frame.loc[top_roots].copy()

    remaining = frame.drop(index=top_roots, errors="ignore")
    if not remaining.empty:
        plotted.loc[-1] = remaining.sum(axis=0)

    return plotted


def method_label(method: str) -> str:
    return method.removeprefix("EF v3.1 - ") if method.startswith("EF v3.1 - ") else method


def method_unit(method: str) -> str:
    import json

    from trails.lcia import _get_lcia_methods_filepath

    filepath = _get_lcia_methods_filepath(EI_VERSION)
    with open(filepath) as handle:
        for row in json.load(handle):
            if " - ".join(row.get("name", [])) == method:
                return clean_text(row.get("unit")) or "impact units"

    if "(" in method and method.endswith(")"):
        return method.rsplit("(", 1)[-1].removesuffix(")")
    return "impact units"


def wrapped(text: str, width: int) -> str:
    words = str(text).split()
    lines: list[str] = []
    current = ""
    for word in words:
        candidate = word if not current else f"{current} {word}"
        if len(candidate) <= width:
            current = candidate
        else:
            if current:
                lines.append(current)
            current = word
    if current:
        lines.append(current)
    return "<br>".join(lines)


def area_mode_slug(stacked: bool) -> str:
    return "stacked" if stacked else "unstacked"


def hex_to_rgba(color: str, alpha: float) -> str:
    text = str(color).lstrip("#")
    red = int(text[0:2], 16)
    green = int(text[2:4], 16)
    blue = int(text[4:6], 16)
    return f"rgba({red}, {green}, {blue}, {alpha:.3f})"


def short_label(text: str) -> str:
    text = str(text)
    limit = int(LEGEND_LABEL_MAX_CHARS)
    if len(text) <= limit:
        return text
    return text[: max(0, limit - 3)].rstrip() + "..."


def reference_product_label(trails: Trails, activity_index: int) -> str:
    for metadata_by_index in trails.activity_indices.values():
        meta = metadata_by_index.get(int(activity_index))
        if meta:
            product = clean_text(meta.get("reference product"))
            return product or f"activity {activity_index}"
    return f"activity {activity_index}"


def contribution_palette() -> list[str]:
    return [
        "#636EFA",
        "#EF553B",
        "#00CC96",
        "#AB63FA",
        "#FFA15A",
        "#19D3F3",
        "#FF6692",
        "#B6E880",
        "#FF97FF",
        "#FECB52",
    ]


def routing_kind(routing: str) -> str:
    return "foreground" if routing.startswith("foreground") else routing


def add_manual_legend(fig: go.Figure, entries: list[tuple[str, str, str]]) -> None:
    max_entries = max(1, int(LEGEND_COLUMNS) * int(LEGEND_MAX_ROWS))
    if len(entries) > max_entries and len(entries) > 3:
        entries = entries[: max_entries - 3] + entries[-3:]
    else:
        entries = entries[:max_entries]
    annotations = []
    column_x = [0.0, 0.53]
    y_start = 1.27
    row_step = 0.038

    for position, (name, color, marker) in enumerate(entries):
        column = position // int(LEGEND_MAX_ROWS)
        row = position % int(LEGEND_MAX_ROWS)
        if column >= int(LEGEND_COLUMNS):
            break
        if marker == "dash":
            marker_text = f'<span style="color:{color}">- - -</span>'
        elif marker == "line":
            marker_text = f'<span style="color:{color}">&#9473;&#9473;&#9473;</span>'
        else:
            marker_text = f'<span style="color:{color}">&#9632;</span>'
        annotations.append(
            dict(
                x=column_x[column],
                y=y_start - row * row_step,
                xref="paper",
                yref="paper",
                xanchor="left",
                yanchor="middle",
                align="left",
                showarrow=False,
                text=f"{marker_text} {short_label(name)}",
                font=dict(size=12, color="#1f3557"),
            )
        )

    fig.update_layout(showlegend=False, annotations=annotations)


def figure_output_paths(*, case_key: str, method: str, stacked: bool) -> tuple[Path, Path]:
    figure_dir = OUTPUT_DIR / case_key
    figure_dir.mkdir(parents=True, exist_ok=True)
    stem = f"{slug(method, 100)}_routing_comparison_{area_mode_slug(stacked)}"
    html_path = figure_dir / f"{stem}.html"
    png_path = figure_dir / f"{stem}.png"
    return html_path, png_path


def rgba_tuple(color: str, alpha: float) -> tuple[float, float, float, float]:
    text = str(color).lstrip("#")
    return (
        int(text[0:2], 16) / 255,
        int(text[2:4], 16) / 255,
        int(text[4:6], 16) / 255,
        alpha,
    )


def positive_y_range(values: list[np.ndarray | pd.Series]) -> list[float]:
    arrays = [np.asarray(value, dtype=float).ravel() for value in values]
    finite_arrays = [array[np.isfinite(array)] for array in arrays if array.size]
    if not finite_arrays:
        return [0.0, 1.0]
    finite_values = np.concatenate(finite_arrays)
    if finite_values.size == 0:
        return [0.0, 1.0]
    y_max = max(0.0, float(finite_values.max()))
    if y_max <= 0:
        return [0.0, 1.0]
    return [0.0, y_max * 1.1]


def write_case_png(*, case_key: str, method: str, stacked: bool, path: Path) -> None:
    import os

    os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT_DIR / ".matplotlib"))
    Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

    import matplotlib

    matplotlib.use("Agg", force=True)
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch

    foreground = results[(case_key, method, "foreground_depth_1")]
    adaptive = results[(case_key, method, "adaptive")]
    contributions = top_root_contributions(adaptive)
    palette = contribution_palette()
    unit = method_unit(adaptive.method)

    figure, axis = plt.subplots(figsize=(WIDTH / 100, HEIGHT / 100), dpi=100)
    figure.patch.set_facecolor("white")
    cumulative_axis = axis.twinx()
    legend_handles = []
    legend_labels = []

    years = np.asarray(contributions.columns, dtype=float)
    baseline = np.zeros_like(years, dtype=float)
    for position, (root_id, values) in enumerate(contributions.iterrows()):
        color = palette[position % len(palette)]
        label = "Other activities" if int(root_id) == -1 else reference_product_label(trails, int(root_id))
        annual = values.to_numpy(dtype=float)
        if stacked:
            next_baseline = baseline + annual
            axis.fill_between(years, baseline, next_baseline, color=rgba_tuple(color, 0.38), linewidth=0)
            baseline = next_baseline
        else:
            axis.fill_between(years, 0, annual, color=rgba_tuple(color, 0.38), linewidth=0)
        legend_handles.append(Patch(facecolor=rgba_tuple(color, 0.75), edgecolor="none"))
        legend_labels.append(short_label(label))

    for result in [foreground, adaptive]:
        kind = routing_kind(result.routing)
        color = ROUTING_CASE_COLORS[kind]
        label = ROUTING_CASE_LABELS[kind]
        line_width = 3.8 if kind == "adaptive" else 3.2
        cumulative_axis.plot(
            result.cumulative.index,
            result.cumulative.values,
            color=color,
            linewidth=line_width,
        )
        legend_handles.append(Line2D([0], [0], color=color, linewidth=line_width))
        legend_labels.append(short_label(f"{label} (cumul.)"))

    cumulative_axis.axhline(adaptive.static_score, color=ROUTING_CASE_COLORS["static"], linewidth=2.4, linestyle="--")
    legend_handles.append(Line2D([0], [0], color=ROUTING_CASE_COLORS["static"], linewidth=2.4, linestyle="--"))
    legend_labels.append(short_label("Static score (2025)"))
    axis.axvline(REFERENCE_YEAR, color="black", alpha=0.45, linewidth=1.0, linestyle="--", zorder=4)
    axis.annotate(
        "reference year",
        xy=(REFERENCE_YEAR, 1.0),
        xycoords=("data", "axes fraction"),
        xytext=(3, -4),
        textcoords="offset points",
        ha="left",
        va="top",
        fontsize=7,
        color="black",
        alpha=0.65,
    )

    axis.set_xlabel("Year", fontsize=10, color="#1f3557")
    axis.set_ylabel(f"Annual impact ({unit})", fontsize=10, color="#1f3557")
    cumulative_axis.set_ylabel(f"Cumulative / static impact ({unit})", fontsize=10, color="#1f3557", labelpad=8)
    axis.set_xticks(np.arange(X_TICK_START, PLOT_YEAR_END + 1, X_TICK_YEARS))
    axis.set_xlim(PLOT_YEAR_START, PLOT_YEAR_END)
    primary_y_values = [baseline] if stacked else [contributions.to_numpy(dtype=float)]
    primary_y_range = positive_y_range(primary_y_values)
    axis.set_ylim(primary_y_range)
    secondary_y_range = positive_y_range([
        foreground.cumulative,
        adaptive.cumulative,
        np.array([adaptive.static_score], dtype=float),
    ])
    cumulative_axis.set_ylim(secondary_y_range)
    axis.set_axisbelow(True)
    axis.grid(True, color="#e6eef8")
    cumulative_axis.grid(False)
    for chart_axis in [axis, cumulative_axis]:
        chart_axis.tick_params(axis="y", labelsize=10, colors="#1f3557")
        chart_axis.tick_params(axis="x", labelsize=8, colors="#1f3557")
        chart_axis.spines["top"].set_visible(False)

    figure.subplots_adjust(left=0.13, right=0.80, top=0.66, bottom=0.14)
    max_entries = int(LEGEND_COLUMNS) * int(LEGEND_MAX_ROWS)
    if len(legend_handles) > max_entries and len(legend_handles) > 3:
        visible_handles = legend_handles[: max_entries - 3] + legend_handles[-3:]
        visible_labels = legend_labels[: max_entries - 3] + legend_labels[-3:]
    else:
        visible_handles = legend_handles[:max_entries]
        visible_labels = legend_labels[:max_entries]
    figure.legend(
        visible_handles,
        visible_labels,
        loc="lower left",
        bbox_to_anchor=(0.08, 0.72, 0.86, 0.26),
        ncol=2,
        frameon=False,
        fontsize=7.5,
        handlelength=1.6,
        columnspacing=0.6,
        handletextpad=0.5,
        labelcolor="#1f3557",
    )

    started = time.perf_counter()
    figure.savefig(path, dpi=100 * PNG_SCALE, facecolor="white")
    plt.close(figure)
    elapsed = time.perf_counter() - started
    print(f"  done in {elapsed:.1f}s", flush=True)


def make_case_plot(case_key: str, method: str, *, stacked: bool) -> go.Figure:
    foreground = results[(case_key, method, "foreground_depth_1")]
    adaptive = results[(case_key, method, "adaptive")]
    contributions = top_root_contributions(adaptive)
    palette = contribution_palette()
    unit = method_unit(adaptive.method)

    fig = go.Figure()
    legend_entries: list[tuple[str, str, str]] = []

    for position, (root_id, values) in enumerate(contributions.iterrows()):
        color = palette[position % len(palette)]
        label = "Other activities" if int(root_id) == -1 else reference_product_label(trails, int(root_id))
        fig.add_trace(
            go.Scatter(
                x=values.index,
                y=values.values,
                mode="lines",
                stackgroup="annual" if stacked else None,
                fill="tonexty" if stacked else "tozeroy",
                line=dict(color=color, width=0),
                fillcolor=hex_to_rgba(color, 0.38),
                name=label,
                hovertemplate="<b>%{x}</b><br>Annual impact: %{y:.6g}<extra>" + label + "</extra>",
                showlegend=False,
            )
        )
        legend_entries.append((label, color, "square"))

    for result in [foreground, adaptive]:
        kind = routing_kind(result.routing)
        color = ROUTING_CASE_COLORS[kind]
        label = ROUTING_CASE_LABELS[kind]
        width = 5.2 if kind == "adaptive" else 4.4
        fig.add_trace(
            go.Scatter(
                x=result.cumulative.index,
                y=result.cumulative.values,
                mode="lines",
                name=f"{label} cumulative",
                line=dict(color=color, width=width),
                yaxis="y2",
                showlegend=False,
                hovertemplate="<b>%{x}</b><br>Cumulative impact: %{y:.6g}<extra></extra>",
            )
        )
        legend_entries.append((f"{label} cumulative", color, "line"))

    fig.add_trace(
        go.Scatter(
            x=[PLOT_YEAR_START, PLOT_YEAR_END],
            y=[adaptive.static_score, adaptive.static_score],
            mode="lines",
            name="Static score",
            line=dict(color="#dc2626", width=3, dash="dash"),
            yaxis="y2",
            showlegend=False,
            hovertemplate="Static score: %{y:.6g}<extra></extra>",
        )
    )
    legend_entries.append(("Static score (2025)", "#dc2626", "dash"))
    fig.add_vline(
        x=REFERENCE_YEAR,
        line=dict(color="rgba(0, 0, 0, 0.45)", width=1.0, dash="dash"),
    )
    fig.add_annotation(
        x=REFERENCE_YEAR,
        y=1.0,
        xref="x",
        yref="paper",
        xanchor="left",
        yanchor="top",
        xshift=3,
        yshift=-4,
        text="reference year",
        showarrow=False,
        font=dict(size=7, color="rgba(0, 0, 0, 0.65)"),
    )

    fig.update_layout(
        width=WIDTH,
        height=HEIGHT,
        title=dict(text="", x=0.5, xanchor="center"),
        font=dict(size=18, color="#1f3557"),
        margin=dict(l=65, r=65, t=135, b=55),
    )
    fig.update_xaxes(
        title_text="Year",
        range=[PLOT_YEAR_START, PLOT_YEAR_END],
        tick0=X_TICK_START,
        dtick=X_TICK_YEARS,
        tickmode="linear",
        tickfont=dict(size=14),
        title_font=dict(size=14),
        zeroline=False,
        showgrid=True,
        gridcolor="#e6eef8",
        layer="below traces",
    )
    primary_y_values = [contributions.sum(axis=0)] if stacked else [contributions.to_numpy(dtype=float)]
    primary_y_range = positive_y_range(primary_y_values)
    secondary_y_range = positive_y_range([
        foreground.cumulative,
        adaptive.cumulative,
        np.array([adaptive.static_score], dtype=float),
    ])

    fig.update_yaxes(
        title_text=f"Annual impact ({unit})",
        tickfont=dict(size=14),
        title_font=dict(size=14),
        zeroline=True,
        zerolinecolor="#64748b",
        showgrid=True,
        gridcolor="#e6eef8",
        layer="below traces",
    )
    fig.update_yaxes(range=primary_y_range)
    fig.update_layout(
        yaxis2=dict(
            title_text=f"Cumulative / static impact ({unit})",
            overlaying="y",
            side="right",
            tickfont=dict(size=14),
            title_font=dict(size=14),
            zeroline=True,
            zerolinecolor="#64748b",
            showgrid=False,
            range=secondary_y_range,
        )
    )
    add_manual_legend(fig, legend_entries)
    return fig


figures: dict[tuple[str, str, str], go.Figure] = {}
png_jobs: list[tuple[str, str, bool, Path]] = []
png_paths: list[Path] = []

for case_key in CASE_KEYS:
    for method in EF_METHODS:
        for stacked in STACKED_AREA_MODES:
            fig = make_case_plot(case_key, method, stacked=stacked)
            figures[(case_key, method, area_mode_slug(stacked))] = fig
            html_path, png_path = figure_output_paths(
                case_key=case_key, method=method, stacked=stacked
            )
            if WRITE_HTML:
                fig.write_html(html_path, include_plotlyjs="cdn")
            if WRITE_PNG:
                png_jobs.append((case_key, method, stacked, png_path))
                png_paths.append(png_path)

print("PNG figures:", len(png_paths))
for path in png_paths:
    print(path)

if WRITE_PNG:
    for index, (case_key, method, stacked, path) in enumerate(png_jobs, start=1):
        print(f"Writing PNG {index}/{len(png_jobs)}: {path}", flush=True)
        write_case_png(case_key=case_key, method=method, stacked=stacked, path=path)

if figures:
    display(next(iter(figures.values())))

## Inspect Results

Use these compact tables to compare static, foreground-only temporal, and adaptive temporal results.

In [ ]:
summary = pd.read_csv(RESULTS_CSV)
columns = [
    "case",
    "method",
    "static_score",
    "temporal_cumulative_foreground_depth_1",
    "temporal_cumulative_adaptive",
    "adaptive_minus_foreground",
    "adaptive_relative_to_foreground",
    "graph_nodes_foreground_depth_1",
    "graph_nodes_adaptive",
    "deepest_level_foreground_depth_1",
    "deepest_level_adaptive",
    "solver_used_foreground_depth_1",
    "solver_used_adaptive",
    "routing_seconds_foreground_depth_1",
    "routing_seconds_adaptive",
]
display(summary[columns])